# Paper #14: Hinode/SOT Implementation / Hinode/SOT 구현
Tsuneta et al. (2008) — Solar Optical Telescope의 핵심 광학 및 편광 측정 원리 구현

이 노트북은 Hinode/SOT 기기의 핵심 개념을 구현합니다:
This notebook implements the core concepts of the Hinode/SOT instrument:

1. 회절 한계 분해능과 Airy 패턴 / Diffraction-limited resolution and Airy pattern
2. Strehl ratio와 Maréchal criterion / Strehl ratio and Maréchal criterion
3. PMU Stokes 편광 변조 및 복조 / PMU Stokes polarization modulation and demodulation
4. Dopplergram 유도 (4-image method) / Dopplergram derivation
5. Longitudinal Magnetogram 유도 / Longitudinal magnetogram derivation
6. 영상 안정화 시스템 시뮬레이션 / Image stabilization system simulation
7. SOT vs 지상 관측 비교 / SOT vs ground-based comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from matplotlib.gridspec import GridSpec

# Standard plot settings
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("All imports successful.")

## Part 1: 회절 한계 분해능과 Airy 패턴 / Diffraction-Limited Resolution and Airy Pattern

SOT는 50 cm 구경으로 회절 한계(0.2–0.3 arcsec) 분해능을 달성합니다.
SOT achieves diffraction-limited (0.2–0.3 arcsec) resolution with its 50 cm aperture.

$$\theta = 1.22 \frac{\lambda}{D}$$

Airy 패턴의 강도 분포 / Airy pattern intensity distribution:

$$I(\theta) = I_0 \left[ \frac{2 J_1(x)}{x} \right]^2, \quad x = \frac{\pi D \theta}{\lambda}$$

In [ ]:
# =============================================================
# Diffraction-Limited Resolution: SOT Airy Pattern
# =============================================================

D_sot = 0.50  # SOT aperture in meters


def diffraction_limit_arcsec(wavelength_nm, aperture_m):
    """Compute diffraction limit in arcsec.

    Args:
        wavelength_nm: Wavelength in nm.
        aperture_m: Aperture diameter in meters.

    Returns:
        Angular resolution in arcsec.
    """
    theta_rad = 1.22 * wavelength_nm * 1e-9 / aperture_m
    return theta_rad * 206265  # rad to arcsec


def airy_pattern(theta_arcsec, wavelength_nm, aperture_m):
    """Compute normalized Airy pattern intensity.

    Args:
        theta_arcsec: Angular offset array in arcsec.
        wavelength_nm: Wavelength in nm.
        aperture_m: Aperture diameter in meters.

    Returns:
        Normalized intensity array.
    """
    theta_rad = theta_arcsec / 206265
    x = np.pi * aperture_m * theta_rad / (wavelength_nm * 1e-9)
    # Handle x=0 singularity
    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(x == 0, 1.0, (2 * special.j1(x) / x) ** 2)
    return result


# Compute resolution at key wavelengths
wavelengths = [388.3, 430.5, 500.0, 557.6, 630.2, 656.3]
labels = ["CN 388", "G-band 430", "500 nm", "Fe I 558", "Fe I 630", "Hα 656"]

print("SOT Diffraction-Limited Resolution (D = {:.0f} cm)".format(D_sot * 100))
print("=" * 50)
for wl, lbl in zip(wavelengths, labels):
    res = diffraction_limit_arcsec(wl, D_sot)
    print("  {:12s}: λ = {:6.1f} nm → θ = {:.3f}\"".format(lbl, wl, res))

# Compare with ground-based telescopes
print("\nComparison with other telescopes:")
telescopes = [("SOT", 0.50), ("SST", 1.0), ("DKIST", 4.0)]
for name, D in telescopes:
    res = diffraction_limit_arcsec(500.0, D)
    print("  {:6s} (D={:.1f} m): θ = {:.3f}\" at 500 nm".format(name, D, res))

# Plot Airy patterns
theta = np.linspace(-0.8, 0.8, 2000)
colors = ["#2196F3", "#4CAF50", "#FF9800"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Airy patterns at different wavelengths for SOT
ax1 = axes[0]
for wl, lbl, c in zip([388.3, 500.0, 656.3],
                       ["CN 388 nm", "500 nm", "Hα 656 nm"],
                       colors):
    I_airy = airy_pattern(theta, wl, D_sot)
    ax1.plot(theta, I_airy, color=c, linewidth=2, label=lbl)
    res = diffraction_limit_arcsec(wl, D_sot)
    ax1.axvline(x=res, color=c, linestyle=":", alpha=0.5)
    ax1.axvline(x=-res, color=c, linestyle=":", alpha=0.5)

ax1.set_xlabel("Angular offset [arcsec]")
ax1.set_ylabel("Normalized Intensity")
ax1.set_title("SOT Airy Patterns (D = 50 cm)")
ax1.legend()
ax1.set_ylim(-0.02, 1.05)

# Right: Comparison of SOT vs SST vs DKIST at 500 nm
ax2 = axes[1]
for (name, D), c in zip(telescopes, colors):
    I_airy = airy_pattern(theta, 500.0, D)
    ax2.plot(theta, I_airy, color=c, linewidth=2,
             label="{} (D={:.1f} m)".format(name, D))

ax2.set_xlabel("Angular offset [arcsec]")
ax2.set_ylabel("Normalized Intensity")
ax2.set_title("Telescope Comparison at 500 nm")
ax2.legend()
ax2.set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.show()

## Part 2: Strehl Ratio와 Maréchal Criterion / Strehl Ratio and Maréchal Criterion

Strehl ratio는 실제 PSF 최대값과 이상적 PSF 최대값의 비율입니다.
Strehl ratio is the ratio of the actual PSF peak to the ideal PSF peak.

$$S = \frac{I_{\text{peak, actual}}}{I_{\text{peak, ideal}}} \approx \exp\left[-\left(\frac{2\pi \sigma}{\lambda}\right)^2\right]$$

**Maréchal criterion**: $S \geq 0.8$ ↔ $\sigma \leq \lambda/14$

SOT 달성값: OTA Strehl > 0.9, FPP Strehl ≈ 0.9, Combined > 0.8
SOT achieved: OTA Strehl > 0.9, FPP Strehl ≈ 0.9, Combined > 0.8

In [ ]:
# =============================================================
# Strehl Ratio and Wavefront Error Analysis
# =============================================================

def strehl_ratio(sigma_nm, wavelength_nm):
    """Compute Strehl ratio from wavefront error.

    Args:
        sigma_nm: RMS wavefront error in nm.
        wavelength_nm: Wavelength in nm.

    Returns:
        Strehl ratio (0 to 1).
    """
    return np.exp(-(2 * np.pi * sigma_nm / wavelength_nm) ** 2)


# Wavefront error range
sigma = np.linspace(0, 80, 500)  # nm rms

# Strehl at different wavelengths
wavelengths_strehl = [388.3, 500.0, 630.2]
colors = ["#9C27B0", "#2196F3", "#E91E63"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Strehl vs WFE
ax1 = axes[0]
for wl, c in zip(wavelengths_strehl, colors):
    S = strehl_ratio(sigma, wl)
    ax1.plot(sigma, S, color=c, linewidth=2, label="λ = {:.0f} nm".format(wl))
    # Mark Marechal criterion
    sigma_crit = wl / 14
    ax1.axvline(x=sigma_crit, color=c, linestyle=":", alpha=0.5)

ax1.axhline(y=0.8, color="black", linestyle="--", alpha=0.5, label="Maréchal S=0.8")

# Mark SOT performance regions
ax1.axhspan(0.9, 1.0, alpha=0.1, color="green", label="SOT OTA region")
ax1.axhspan(0.8, 0.9, alpha=0.1, color="orange", label="SOT combined")

ax1.set_xlabel("RMS Wavefront Error σ [nm]")
ax1.set_ylabel("Strehl Ratio S")
ax1.set_title("Strehl Ratio vs Wavefront Error")
ax1.legend(fontsize=8, loc="lower left")
ax1.set_ylim(0, 1.05)

# Right: PSF comparison (ideal vs aberrated)
ax2 = axes[1]
theta_psf = np.linspace(-0.6, 0.6, 1000)

# Ideal PSF
psf_ideal = airy_pattern(theta_psf, 500.0, D_sot)

# Approximate effect of wavefront error on PSF
# Aberrated PSF: peak reduced by Strehl, energy spreads to wider angles
wfe_cases = [0, 20, 35.7, 50]  # nm rms
psf_colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

for wfe, c in zip(wfe_cases, psf_colors):
    S = strehl_ratio(wfe, 500.0)
    # Simplified: coherent part (Airy) + incoherent halo
    halo_width = 0.3 * (1 + wfe / 30)  # arcsec, broadens with WFE
    halo = (1 - S) * np.exp(-theta_psf ** 2 / (2 * halo_width ** 2))
    halo = halo / halo.max() * (1 - S) if halo.max() > 0 else halo
    psf = S * psf_ideal + halo
    ax2.plot(theta_psf, psf, color=c, linewidth=1.5,
             label="σ={:.0f} nm (S={:.2f})".format(wfe, S))

ax2.set_xlabel("Angular offset [arcsec]")
ax2.set_ylabel("Normalized Intensity")
ax2.set_title("PSF Degradation with Wavefront Error (500 nm)")
ax2.legend(fontsize=9)
ax2.set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.show()

# Numerical results
print("SOT Wavefront Error Budget:")
print("  Maréchal limit at 500 nm: σ ≤ {:.1f} nm".format(500.0 / 14))
print("  OTA Strehl > 0.9 → σ_OTA < {:.1f} nm".format(
    500.0 * np.sqrt(-np.log(0.9)) / (2 * np.pi)))
print("  Combined Strehl > 0.8 → σ_total < {:.1f} nm".format(
    500.0 * np.sqrt(-np.log(0.8)) / (2 * np.pi)))
print("  If OTA σ=20 nm and FPP σ=25 nm:")
print("    σ_total = {:.1f} nm → S = {:.3f}".format(
    np.sqrt(20**2 + 25**2),
    strehl_ratio(np.sqrt(20**2 + 25**2), 500.0)))

## Part 3: PMU Stokes 편광 변조 / PMU Stokes Polarization Modulation

SOT의 PMU(Polarization Modulator Unit)는 연속 회전 파장판(T = 1.6 s)입니다.
SOT's PMU is a continuously rotating waveplate (T = 1.6 s).

$$I_{\text{measured}} = \frac{1}{2}\left[I + Q\cos(4\phi) + U\sin(4\phi) + V\sin(2\phi)\right]$$

여기서 $\phi = 2\pi t / T$이고, PMU 1회전당 16 등간격 샘플로 복조합니다.
where $\phi = 2\pi t / T$, and demodulation uses 16 equally-spaced samples per revolution.

In [ ]:
# =============================================================
# PMU Stokes Modulation and Demodulation
# =============================================================

T_pmu = 1.6  # PMU revolution period in seconds
n_samples = 16  # samples per revolution


def pmu_modulate(phi, I, Q, U, V):
    """Simulate PMU intensity modulation.

    Args:
        phi: PMU rotation angle in radians.
        I, Q, U, V: Input Stokes parameters.

    Returns:
        Measured intensity.
    """
    return 0.5 * (I + Q * np.cos(4 * phi) + U * np.sin(4 * phi)
                  + V * np.sin(2 * phi))


def pmu_demodulate(measurements, phi_samples):
    """Demodulate 16-sample measurements to recover Stokes parameters.

    Args:
        measurements: Array of 16 intensity measurements.
        phi_samples: Array of 16 PMU angles.

    Returns:
        Recovered (I, Q, U, V).
    """
    n = len(measurements)
    I_rec = 2.0 * np.mean(measurements)
    Q_rec = 4.0 * np.mean(measurements * np.cos(4 * phi_samples))
    U_rec = 4.0 * np.mean(measurements * np.sin(4 * phi_samples))
    V_rec = 4.0 * np.mean(measurements * np.sin(2 * phi_samples))
    return I_rec, Q_rec, U_rec, V_rec


# Time array for smooth curve
t_fine = np.linspace(0, T_pmu, 1000)
phi_fine = 2 * np.pi * t_fine / T_pmu

# 16 sample points
t_samples = np.linspace(0, T_pmu, n_samples, endpoint=False)
phi_samples = 2 * np.pi * t_samples / T_pmu

# Input Stokes parameters (typical solar observation)
I_in, Q_in, U_in, V_in = 1.0, 0.05, -0.03, 0.08

# Modulated signal
I_mod_fine = pmu_modulate(phi_fine, I_in, Q_in, U_in, V_in)
I_mod_samples = pmu_modulate(phi_samples, I_in, Q_in, U_in, V_in)

# Demodulate
I_rec, Q_rec, U_rec, V_rec = pmu_demodulate(I_mod_samples, phi_samples)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Top-left: full modulated signal
ax = axes[0, 0]
ax.plot(t_fine * 1000, I_mod_fine, "b-", linewidth=1.5, label="Modulated I(t)")
ax.plot(t_samples * 1000, I_mod_samples, "ro", markersize=8, label="16 samples")
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Measured Intensity")
ax.set_title("PMU Modulated Signal (T = {:.1f} s)".format(T_pmu))
ax.legend()

# Top-right: individual Stokes contributions
ax = axes[0, 1]
ax.plot(t_fine * 1000, 0.5 * I_in * np.ones_like(t_fine), "k--",
        label="I/2 (DC)", alpha=0.5)
ax.plot(t_fine * 1000, 0.5 * Q_in * np.cos(4 * phi_fine), "b-",
        label="Q·cos(4φ) [T/4]", linewidth=1.5)
ax.plot(t_fine * 1000, 0.5 * U_in * np.sin(4 * phi_fine), "g-",
        label="U·sin(4φ) [T/4]", linewidth=1.5)
ax.plot(t_fine * 1000, 0.5 * V_in * np.sin(2 * phi_fine), "r-",
        label="V·sin(2φ) [T/2]", linewidth=1.5)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Intensity Contribution")
ax.set_title("Individual Stokes Modulation Components")
ax.legend(fontsize=8)

# Bottom-left: modulation frequencies
ax = axes[1, 0]
freqs = np.fft.rfftfreq(len(t_fine), d=t_fine[1] - t_fine[0])
fft_signal = np.abs(np.fft.rfft(I_mod_fine - I_mod_fine.mean()))
ax.stem(freqs[:20], fft_signal[:20], linefmt="b-", markerfmt="bo", basefmt="k-")
# Mark expected frequencies
f_V = 2.0 / T_pmu  # V modulation at 2/T
f_QU = 4.0 / T_pmu  # Q,U modulation at 4/T
ax.axvline(x=f_V, color="red", linestyle="--", alpha=0.7,
           label="V: 2/T = {:.1f} Hz".format(f_V))
ax.axvline(x=f_QU, color="green", linestyle="--", alpha=0.7,
           label="Q,U: 4/T = {:.1f} Hz".format(f_QU))
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Amplitude")
ax.set_title("Frequency Spectrum of Modulated Signal")
ax.set_xlim(0, 5)
ax.legend(fontsize=9)

# Bottom-right: recovery accuracy
ax = axes[1, 1]
stokes_names = ["I", "Q", "U", "V"]
input_vals = [I_in, Q_in, U_in, V_in]
recovered_vals = [I_rec, Q_rec, U_rec, V_rec]
x_pos = np.arange(4)
width = 0.35
bars1 = ax.bar(x_pos - width / 2, input_vals, width, label="Input",
               color="#2196F3", edgecolor="black")
bars2 = ax.bar(x_pos + width / 2, recovered_vals, width, label="Recovered",
               color="#FF9800", edgecolor="black")
ax.set_xticks(x_pos)
ax.set_xticklabels(stokes_names)
ax.set_ylabel("Value")
ax.set_title("Stokes Recovery (Noiseless)")
ax.legend()

plt.tight_layout()
plt.show()

print("Demodulation results:")
print("  Input:     I={:.4f}  Q={:.4f}  U={:.4f}  V={:.4f}".format(
    I_in, Q_in, U_in, V_in))
print("  Recovered: I={:.4f}  Q={:.4f}  U={:.4f}  V={:.4f}".format(
    I_rec, Q_rec, U_rec, V_rec))
print("  Error:     I={:.2e}  Q={:.2e}  U={:.2e}  V={:.2e}".format(
    abs(I_rec - I_in), abs(Q_rec - Q_in), abs(U_rec - U_in), abs(V_rec - V_in)))

## Part 4: 노이즈 하에서의 Stokes 복조 성능 / Demodulation Performance Under Noise

실제 관측에서는 광자 노이즈가 복조 정밀도를 제한합니다.
In real observations, photon noise limits demodulation accuracy.

PMU의 연속 회전은 비자기 관측에는 완전히 투명하면서도 Q, U, V를 균등 효율(~0.5)로 변조합니다.
The PMU's continuous rotation is transparent to non-magnetic observations while modulating Q, U, V at equal efficiency (~0.5).

In [ ]:
# =============================================================
# Stokes Demodulation with Photon Noise
# =============================================================

np.random.seed(42)

# Test with varying noise levels
noise_levels = [0.0001, 0.001, 0.005, 0.01, 0.05]
n_trials = 1000

results = {name: [] for name in ["σ_I", "σ_Q", "σ_U", "σ_V"]}

for noise_sigma in noise_levels:
    errors = np.zeros((n_trials, 4))
    for trial in range(n_trials):
        noise = np.random.normal(0, noise_sigma, n_samples)
        noisy = I_mod_samples + noise
        Ir, Qr, Ur, Vr = pmu_demodulate(noisy, phi_samples)
        errors[trial] = [Ir - I_in, Qr - Q_in, Ur - U_in, Vr - V_in]
    rms = np.std(errors, axis=0)
    for i, name in enumerate(["σ_I", "σ_Q", "σ_U", "σ_V"]):
        results[name].append(rms[i])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: recovery error vs noise level
ax = axes[0]
for name, marker, c in zip(["σ_I", "σ_Q", "σ_U", "σ_V"],
                            ["o", "s", "^", "D"],
                            ["black", "#2196F3", "#4CAF50", "#E91E63"]):
    ax.loglog(noise_levels, results[name], marker=marker, color=c,
              linewidth=1.5, markersize=8, label=name)

ax.set_xlabel("Measurement Noise σ")
ax.set_ylabel("Stokes Recovery RMS Error")
ax.set_title("Demodulation Accuracy vs Noise (16-sample)")
ax.legend()

# Right: single noisy example
ax = axes[1]
noise_demo = np.random.normal(0, 0.005, n_samples)
noisy_demo = I_mod_samples + noise_demo

ax.plot(t_fine * 1000, I_mod_fine, "b-", linewidth=1, alpha=0.5, label="True signal")
ax.plot(t_samples * 1000, I_mod_samples, "bo", markersize=6, label="Noiseless")
ax.plot(t_samples * 1000, noisy_demo, "r^", markersize=8, label="Noisy (σ=0.005)")
ax.errorbar(t_samples * 1000, noisy_demo, yerr=0.005, fmt="none",
            ecolor="red", alpha=0.3)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Measured Intensity")
ax.set_title("Single PMU Revolution: Noiseless vs Noisy")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Modulation efficiency
print("PMU Modulation Efficiency:")
print("  Design wavelengths: 630.2 nm (1.35 waves), 517.2 nm (1.85 waves)")
print("  Efficiency for Q, U, V: ~0.5 (equally balanced)")
print("  Advantage: equal sensitivity to all magnetic orientations")

## Part 5: Dopplergram 유도 / Dopplergram Derivation

NFI의 4-image method로 Doppler 속도를 추정합니다.
Doppler velocity is estimated using NFI's 4-image method.

$$v_{\text{Doppler}} \propto \frac{(I_1 + I_2) - (I_3 + I_4)}{(I_1 + I_2) + (I_3 + I_4)}$$

- 최적선: Fe I 557.6 nm (Landé g = 0 → 자기장 무감)
- Best line: Fe I 557.6 nm (Landé g = 0 → insensitive to magnetic field)
- rms 노이즈: ~30 m/s (4-image)

In [ ]:
# =============================================================
# SOT/NFI Dopplergram: 4-Image Method
# =============================================================

def gaussian_line(wavelength, lambda0, sigma, depth):
    """Compute Gaussian absorption line profile.

    Args:
        wavelength: Wavelength array in nm.
        lambda0: Line center in nm.
        sigma: Line width in nm.
        depth: Line depth (0-1).

    Returns:
        Normalized intensity profile.
    """
    return 1.0 - depth * np.exp(-0.5 * ((wavelength - lambda0) / sigma) ** 2)


def dopplergram_4image(I1, I2, I3, I4):
    """Compute Dopplergram signal from 4 filtergram images.

    Images are at equally-spaced wavelength positions across a spectral line.
    I1, I2 on the blue wing; I3, I4 on the red wing.

    Args:
        I1, I2, I3, I4: Intensities at 4 wavelength positions.

    Returns:
        Doppler signal proportional to velocity.
    """
    return ((I1 + I2) - (I3 + I4)) / ((I1 + I2) + (I3 + I4))


# Fe I 557.6 nm line parameters
lambda0 = 557.6  # nm
line_sigma = 0.004  # nm
line_depth = 0.6

# NFI Lyot filter bandwidth ~ 95 mA at 630 nm, scale for 557.6 nm
# Sample positions: +/- 40 mA and +/- 80 mA from line center
offsets_mA = np.array([-80, -40, 40, 80])
offsets_nm = offsets_mA * 1e-4

# Wavelength grid for plotting
wav = np.linspace(557.55, 557.65, 1000)

# Test Doppler velocities
velocities = np.linspace(-3, 3, 200)  # km/s
c_km = 3e5  # speed of light in km/s

doppler_signals = []
for v in velocities:
    shift = lambda0 * v / c_km
    positions = lambda0 + offsets_nm
    intensities = gaussian_line(positions, lambda0 + shift, line_sigma, line_depth)
    signal = dopplergram_4image(*intensities)
    doppler_signals.append(signal)

doppler_signals = np.array(doppler_signals)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Line profile with sample positions
ax = axes[0, 0]
profile_rest = gaussian_line(wav, lambda0, line_sigma, line_depth)
profile_shifted = gaussian_line(wav, lambda0 + lambda0 * 1.5 / c_km,
                                line_sigma, line_depth)
ax.plot(wav, profile_rest, "b-", linewidth=2, label="Rest (v = 0)")
ax.plot(wav, profile_shifted, "r--", linewidth=2, label="v = 1.5 km/s")

sample_wavs = lambda0 + offsets_nm
sample_Is = gaussian_line(sample_wavs, lambda0, line_sigma, line_depth)
colors_samp = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]
for j, (sw, si, c) in enumerate(zip(sample_wavs, sample_Is, colors_samp)):
    ax.axvline(x=sw, color=c, linestyle=":", alpha=0.5)
    ax.plot(sw, si, "o", color=c, markersize=10, zorder=5,
            label="I{} ({:+.0f} mÅ)".format(j + 1, offsets_mA[j]))

ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Normalized Intensity")
ax.set_title("Fe I 557.6 nm with NFI Sample Positions")
ax.legend(fontsize=7, ncol=2)

# Top-right: Dopplergram signal vs velocity (calibration curve)
ax = axes[0, 1]
ax.plot(velocities, doppler_signals, "b-", linewidth=2)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
# Mark linear regime
linear_mask = np.abs(velocities) < 1.5
ax.fill_between(velocities[linear_mask], -0.15, 0.15,
                alpha=0.1, color="green", label="Linear regime")
ax.set_xlabel("Doppler Velocity [km/s]")
ax.set_ylabel("Dopplergram Signal")
ax.set_title("Velocity Calibration Curve")
ax.legend()

# Bottom-left: Simulated 2D Dopplergram (synthetic granulation)
ax = axes[1, 0]
np.random.seed(123)
nx, ny = 128, 128
# Create synthetic granulation velocity field
from scipy.ndimage import gaussian_filter
raw = np.random.randn(nx, ny)
v_field = gaussian_filter(raw, sigma=5) * 2.0  # km/s, smoothed

# Convert to Dopplergram signal
doppler_map = np.zeros_like(v_field)
for i in range(nx):
    for j in range(ny):
        shift = lambda0 * v_field[i, j] / c_km
        Is = gaussian_line(sample_wavs, lambda0 + shift, line_sigma, line_depth)
        doppler_map[i, j] = dopplergram_4image(*Is)

# Scale to arcsec (0.08"/pixel for NFI)
extent = [0, nx * 0.08, 0, ny * 0.08]
im = ax.imshow(doppler_map, cmap="RdBu_r", origin="lower", extent=extent,
               vmin=-0.08, vmax=0.08)
plt.colorbar(im, ax=ax, label="Doppler Signal")
ax.set_xlabel('X [arcsec]')
ax.set_ylabel('Y [arcsec]')
ax.set_title("Synthetic NFI Dopplergram")

# Bottom-right: noise analysis
ax = axes[1, 1]
noise_sigmas = np.logspace(-4, -1, 50)
v_rms_list = []
for ns in noise_sigmas:
    v_errors = []
    for _ in range(200):
        Is_clean = gaussian_line(sample_wavs, lambda0, line_sigma, line_depth)
        Is_noisy = Is_clean + np.random.normal(0, ns, 4)
        sig_clean = dopplergram_4image(*Is_clean)
        sig_noisy = dopplergram_4image(*Is_noisy)
        v_errors.append(sig_noisy - sig_clean)
    v_rms_list.append(np.std(v_errors))

# Convert signal rms to velocity rms using linear calibration
# Approximate: dv/dsignal from linear region
slope_idx = np.argmin(np.abs(velocities))
dv_dsig = np.gradient(velocities, doppler_signals)[slope_idx]

ax.loglog(noise_sigmas, np.array(v_rms_list) * abs(dv_dsig) * 1000,
          "b-", linewidth=2)
ax.axhline(y=30, color="red", linestyle="--",
           label="SOT spec: 30 m/s rms")
ax.set_xlabel("Intensity Noise σ_I")
ax.set_ylabel("Velocity Noise [m/s]")
ax.set_title("Dopplergram Velocity Noise vs Intensity Noise")
ax.legend()

plt.tight_layout()
plt.show()

print("Fe I 557.6 nm Dopplergram:")
print("  Landé g = 0 → magnetically insensitive")
print("  4-image method rms noise: ~30 m/s")
print("  Advantage: pure velocity measurement without magnetic contamination")

## Part 6: Longitudinal Magnetogram 유도 / Longitudinal Magnetogram Derivation

시선 방향 자기장은 좌원편광(LCP)과 우원편광(RCP) 강도 차이로 측정합니다.
Line-of-sight magnetic field is measured from the difference between LCP and RCP intensities.

$$M_{\text{long}} \propto \frac{I_R - I_L}{I_R + I_L} \propto \frac{V}{I}$$

- 주요선: Fe I 630.25 nm (광구), Mg I 517.27 nm (저 채층)
- Primary lines: Fe I 630.25 nm (photosphere), Mg I 517.27 nm (low chromosphere)
- rms 노이즈: ~10^15 Mx/pixel (8-image, ~20 s)

In [ ]:
# =============================================================
# SOT/NFI Longitudinal Magnetogram
# =============================================================

def zeeman_splitting_nm(wavelength_nm, g_eff, B_gauss):
    """Compute Zeeman splitting in nm.

    Args:
        wavelength_nm: Central wavelength in nm.
        g_eff: Effective Lande g-factor.
        B_gauss: Magnetic field in Gauss.

    Returns:
        Wavelength splitting in nm.
    """
    return 4.67e-13 * wavelength_nm ** 2 * g_eff * B_gauss


def stokes_V_signal(wavelength, lambda0, sigma, depth, B_gauss, g_eff, gamma=0):
    """Compute Stokes V/I signal for weak-field approximation.

    Args:
        wavelength: Wavelength array in nm.
        lambda0: Line center in nm.
        sigma: Line width in nm.
        depth: Line depth.
        B_gauss: Magnetic field strength in Gauss.
        g_eff: Effective Lande g-factor.
        gamma: Field inclination from LOS (radians).

    Returns:
        Tuple of (I_R, I_L) for right and left circular polarizations.
    """
    dL = zeeman_splitting_nm(lambda0, g_eff, B_gauss) * np.cos(gamma)
    I_R = gaussian_line(wavelength, lambda0 + dL, sigma, depth)
    I_L = gaussian_line(wavelength, lambda0 - dL, sigma, depth)
    return I_R, I_L


# Fe I 630.25 nm parameters
lambda_fe = 630.25  # nm
sigma_fe = 0.004  # nm
depth_fe = 0.65
g_eff_fe = 2.5  # effective Lande g-factor

wav = np.linspace(630.20, 630.30, 1000)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Stokes I and V profiles at different B
ax = axes[0, 0]
B_values = [100, 500, 1000, 2000]
colors_B = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

for B, c in zip(B_values, colors_B):
    I_R, I_L = stokes_V_signal(wav, lambda_fe, sigma_fe, depth_fe, B, g_eff_fe)
    V_over_I = (I_R - I_L) / (I_R + I_L)
    ax.plot(wav, V_over_I, color=c, linewidth=1.5, label="B = {} G".format(B))

ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("V / I")
ax.set_title("Stokes V/I Profile (Fe I 630.25 nm)")
ax.legend()

# Top-right: Magnetogram signal vs B (calibration)
ax = axes[0, 1]
B_range = np.linspace(-3000, 3000, 500)

# NFI measures at line wing position
wing_offset = 0.004  # nm from line center
sample_wav = lambda_fe + wing_offset

mag_signal = []
for B in B_range:
    I_R, I_L = stokes_V_signal(np.array([sample_wav]), lambda_fe,
                                sigma_fe, depth_fe, abs(B), g_eff_fe,
                                gamma=0 if B >= 0 else np.pi)
    mag_signal.append((I_R[0] - I_L[0]) / (I_R[0] + I_L[0]))

ax.plot(B_range, mag_signal, "b-", linewidth=2)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.fill_between(B_range, -0.01, 0.01, where=np.abs(B_range) < 500,
                alpha=0.1, color="green", label="Weak-field regime")
ax.set_xlabel("Magnetic Field B [Gauss]")
ax.set_ylabel("Magnetogram Signal V/I")
ax.set_title("Magnetogram Calibration Curve")
ax.legend()

# Bottom-left: Synthetic 2D magnetogram
ax = axes[1, 0]
np.random.seed(77)
nx, ny = 128, 128

# Create synthetic bipolar active region
x = np.linspace(-5, 5, nx)
y = np.linspace(-5, 5, ny)
X, Y = np.meshgrid(x, y)

# Two magnetic poles (positive and negative)
B_field = (1500 * np.exp(-((X - 1.5) ** 2 + Y ** 2) / 1.5)
           - 1200 * np.exp(-((X + 1.5) ** 2 + Y ** 2) / 2.0))
# Add network field
B_noise = gaussian_filter(np.random.randn(nx, ny), sigma=3) * 100
B_total = B_field + B_noise

# Convert to magnetogram signal
mag_map = np.zeros_like(B_total)
for i in range(nx):
    for j in range(ny):
        B = B_total[i, j]
        I_R, I_L = stokes_V_signal(np.array([sample_wav]), lambda_fe,
                                    sigma_fe, depth_fe, abs(B), g_eff_fe,
                                    gamma=0 if B >= 0 else np.pi)
        mag_map[i, j] = (I_R[0] - I_L[0]) / (I_R[0] + I_L[0])

extent = [0, nx * 0.08, 0, ny * 0.08]  # NFI pixel = 0.08"
im = ax.imshow(mag_map, cmap="RdBu_r", origin="lower", extent=extent,
               vmin=-0.05, vmax=0.05)
plt.colorbar(im, ax=ax, label="V/I Signal")
ax.set_xlabel('X [arcsec]')
ax.set_ylabel('Y [arcsec]')
ax.set_title("Synthetic NFI Longitudinal Magnetogram")

# Bottom-right: Line comparison Fe I 630 vs Mg I 517
ax = axes[1, 1]
lines = [
    ("Fe I 630.25", 630.25, 2.5, "Photosphere"),
    ("Fe I 525.02", 525.02, 3.0, "Photosphere"),
    ("Mg I 517.27", 517.27, 1.75, "Low chromosphere"),
]

B_test = np.linspace(0, 2000, 100)
for name, lam, g, layer in lines:
    split = zeeman_splitting_nm(lam, g, B_test) * 1e4  # to mA
    ax.plot(B_test, split, linewidth=2, label="{} (g={:.1f})".format(name, g))

ax.set_xlabel("Magnetic Field B [Gauss]")
ax.set_ylabel("Zeeman Splitting [mÅ]")
ax.set_title("Zeeman Splitting: SOT/NFI Lines")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("SOT/NFI Magnetogram Specifications:")
print("  Cadence: ~20 s (8-image)")
print("  Noise: ~10^15 Mx/pixel")
print("  Pixel size: 0.08 arcsec")
print("  Fe I 630.25 nm: g_eff = 2.5, photospheric")
print("  Mg I 517.27 nm: g_eff = 1.75, low chromospheric")

## Part 7: 영상 안정화 시스템 시뮬레이션 / Image Stabilization System Simulation

SOT의 correlation tracker + tip-tilt mirror 시스템은 0.007" rms 안정도를 달성합니다.
SOT's correlation tracker + tip-tilt mirror system achieves 0.007" rms stability.

- Correlation tracker: 580 Hz CCD, 11"×11" FOV, 태양립 패턴 추적
- Tip-tilt mirror: piezo 구동, 폐루프 대역폭 < 14 Hz
- 달성 안정도: 0.007" (1σ), 요구 사양의 4배 이상 우수

In [ ]:
# =============================================================
# Image Stabilization System Simulation
# =============================================================

np.random.seed(42)

# Simulation parameters
dt = 1.0 / 580  # CT sampling at 580 Hz
t_total = 2.0  # seconds
t = np.arange(0, t_total, dt)
n_pts = len(t)

# Generate spacecraft jitter (multiple frequency components)
# Low-freq attitude drift + mid-freq reaction wheel + high-freq micro-vibration
jitter_x = (0.05 * np.sin(2 * np.pi * 0.3 * t)  # slow drift
            + 0.02 * np.sin(2 * np.pi * 5.0 * t + 0.5)  # reaction wheel
            + 0.01 * np.sin(2 * np.pi * 25.0 * t + 1.2)  # micro-vib
            + 0.005 * np.sin(2 * np.pi * 80.0 * t)  # high-freq
            + 0.003 * np.random.randn(n_pts))  # random noise

jitter_y = (0.04 * np.sin(2 * np.pi * 0.25 * t + 0.8)
            + 0.015 * np.sin(2 * np.pi * 4.5 * t + 1.0)
            + 0.008 * np.sin(2 * np.pi * 30.0 * t + 0.3)
            + 0.004 * np.sin(2 * np.pi * 75.0 * t + 0.7)
            + 0.003 * np.random.randn(n_pts))

# Simulate closed-loop correction (low-pass filter at 14 Hz bandwidth)
from scipy.signal import butter, filtfilt

# Design low-pass filter at 14 Hz (servo bandwidth)
b, a = butter(2, 14.0 / (580 / 2), btype="low")

# Correction signal = low-passed version of jitter (what servo can track)
correction_x = filtfilt(b, a, jitter_x)
correction_y = filtfilt(b, a, jitter_y)

# Residual jitter after correction
residual_x = jitter_x - correction_x
residual_y = jitter_y - correction_y

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: time series
ax = axes[0, 0]
ax.plot(t * 1000, jitter_x, "r-", alpha=0.5, linewidth=0.5,
        label="Uncorrected ({:.3f}\" rms)".format(np.std(jitter_x)))
ax.plot(t * 1000, residual_x, "b-", linewidth=0.8,
        label="Corrected ({:.4f}\" rms)".format(np.std(residual_x)))
ax.axhline(y=0.03, color="orange", linestyle="--", alpha=0.7, label="Requirement (0.03\")")
ax.axhline(y=-0.03, color="orange", linestyle="--", alpha=0.7)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Jitter X [arcsec]")
ax.set_title("Image Jitter: Before/After Stabilization")
ax.legend(fontsize=8)

# Top-right: power spectrum
ax = axes[0, 1]
freqs_psd = np.fft.rfftfreq(n_pts, d=dt)
psd_raw = np.abs(np.fft.rfft(jitter_x)) ** 2 / n_pts
psd_res = np.abs(np.fft.rfft(residual_x)) ** 2 / n_pts

ax.semilogy(freqs_psd[1:], psd_raw[1:], "r-", alpha=0.5, linewidth=0.5,
            label="Uncorrected")
ax.semilogy(freqs_psd[1:], psd_res[1:], "b-", linewidth=0.8,
            label="Corrected")
ax.axvline(x=14, color="green", linestyle="--", alpha=0.7,
           label="Servo bandwidth (14 Hz)")
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("Power Spectral Density")
ax.set_title("Jitter Power Spectrum")
ax.set_xlim(0.1, 290)
ax.legend(fontsize=9)

# Bottom-left: 2D jitter scatter plot
ax = axes[1, 0]
ax.plot(jitter_x, jitter_y, "r.", markersize=0.5, alpha=0.3, label="Uncorrected")
ax.plot(residual_x, residual_y, "b.", markersize=0.5, alpha=0.5, label="Corrected")

# Draw requirement circle
theta_circle = np.linspace(0, 2 * np.pi, 100)
ax.plot(0.03 * np.cos(theta_circle), 0.03 * np.sin(theta_circle),
        "orange", linestyle="--", linewidth=2, label='Req: 0.03"')
ax.plot(0.007 * np.cos(theta_circle), 0.007 * np.sin(theta_circle),
        "green", linestyle="-", linewidth=2, label='Achieved: 0.007"')

ax.set_xlabel("X [arcsec]")
ax.set_ylabel("Y [arcsec]")
ax.set_title("2D Jitter Distribution")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_xlim(-0.08, 0.08)
ax.set_ylim(-0.08, 0.08)

# Bottom-right: histogram of residuals
ax = axes[1, 1]
r_raw = np.sqrt(jitter_x ** 2 + jitter_y ** 2)
r_res = np.sqrt(residual_x ** 2 + residual_y ** 2)

ax.hist(r_raw, bins=50, alpha=0.5, color="red", density=True,
        label="Uncorrected (rms={:.3f}\")".format(np.std(r_raw)))
ax.hist(r_res, bins=50, alpha=0.7, color="blue", density=True,
        label='Corrected (rms={:.4f}")'.format(np.std(r_res)))
ax.axvline(x=0.03, color="orange", linestyle="--", linewidth=2, label='Req: 0.03"')
ax.axvline(x=0.007, color="green", linestyle="-", linewidth=2, label='Achieved: 0.007"')
ax.set_xlabel("Radial Jitter [arcsec]")
ax.set_ylabel("Probability Density")
ax.set_title("Jitter Amplitude Distribution")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Image Stabilization Performance:")
print("  Uncorrected jitter: X={:.3f}\", Y={:.3f}\" rms".format(
    np.std(jitter_x), np.std(jitter_y)))
print("  Corrected residual: X={:.4f}\", Y={:.4f}\" rms".format(
    np.std(residual_x), np.std(residual_y)))
print("  Requirement: 0.030\"")
print("  Achieved:    0.007\" (4.3x better than requirement)")
print("  Servo bandwidth: < 14 Hz")
print("  CT sampling rate: 580 Hz")

## Part 8: SOT vs 지상 망원경 비교 / SOT vs Ground-Based Telescopes

SOT의 핵심 장점은 대기 없이 안정적이고 연속적인 관측이 가능하다는 것입니다.
SOT's key advantage is stable, continuous observations without atmospheric effects.

지상 관측에서의 atmospheric seeing은 PSF를 시간적으로 변동시키고 장기간 관측을 제한합니다.
Atmospheric seeing at ground-based observatories causes temporal PSF variations and limits long-duration observations.

In [ ]:
# =============================================================
# SOT vs Ground-Based: PSF Stability Comparison
# =============================================================

np.random.seed(42)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: PSF stability over time
ax = axes[0, 0]
n_frames = 100
time_frames = np.arange(n_frames) * 10  # seconds

# SOT: nearly constant Strehl
sot_strehl = 0.85 + 0.02 * np.random.randn(n_frames)
sot_strehl = np.clip(sot_strehl, 0.75, 0.95)

# Ground-based with AO: variable Strehl
seeing_variation = 0.5 + 0.3 * np.sin(2 * np.pi * time_frames / 300)
ground_strehl = 0.4 + 0.3 * np.random.randn(n_frames) * seeing_variation
ground_strehl = np.clip(ground_strehl, 0.05, 0.85)

ax.plot(time_frames, sot_strehl, "b-", linewidth=1.5, label="SOT (space)")
ax.plot(time_frames, ground_strehl, "r-", linewidth=1, alpha=0.7,
        label="Ground + AO")
ax.axhline(y=0.8, color="gray", linestyle="--", alpha=0.5, label="Maréchal")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Strehl Ratio")
ax.set_title("PSF Stability Over Time")
ax.legend()
ax.set_ylim(0, 1)

# Top-right: effective resolution histogram
ax = axes[0, 1]
# SOT: consistent near diffraction limit
sot_res = 0.25 + 0.01 * np.random.randn(1000)
# Ground: broad distribution depending on seeing
ground_res = np.clip(0.15 + np.abs(0.3 * np.random.randn(1000)), 0.1, 1.5)

ax.hist(sot_res, bins=30, alpha=0.7, color="#2196F3", density=True,
        label='SOT (mean={:.2f}")'.format(np.mean(sot_res)))
ax.hist(ground_res, bins=40, alpha=0.5, color="#E91E63", density=True,
        label='Ground (mean={:.2f}")'.format(np.mean(ground_res)))
ax.axvline(x=0.252, color="blue", linestyle="--",
           label='SOT diffraction limit (0.252")')
ax.set_xlabel('Effective Resolution [arcsec]')
ax.set_ylabel("Probability Density")
ax.set_title("Resolution Distribution")
ax.legend(fontsize=8)

# Bottom-left: observing duty cycle
ax = axes[1, 0]
categories = ["SOT\n(sun-sync orbit)", "Ground\n(mid-latitude)",
              "Ground\n(polar, summer)", "Ground\n(network)"]
# Hours per day of useful high-res observation
duty_hours = [22, 4, 10, 16]  # approximate
duty_frac = [h / 24 * 100 for h in duty_hours]
colors_duty = ["#2196F3", "#E91E63", "#FF9800", "#4CAF50"]

bars = ax.bar(range(len(categories)), duty_frac, color=colors_duty,
              edgecolor="black", width=0.6)
for bar, val in zip(bars, duty_frac):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            "{:.0f}%".format(val), ha="center", fontweight="bold")
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylabel("Observing Duty Cycle [%]")
ax.set_title("Observing Time Comparison")
ax.set_ylim(0, 105)

# Bottom-right: comprehensive comparison table
ax = axes[1, 1]
ax.axis("off")
columns = ["Property", "SOT", "SST (1m)", "DKIST (4m)"]
data = [
    ["Aperture", "50 cm", "100 cm", "400 cm"],
    ['Diff. limit (500nm)', '0.25"', '0.13"', '0.03"'],
    ["PSF stability", "Constant", "Variable", "Variable"],
    ["Continuous obs.", "~8 mo/yr", "Daytime", "Daytime"],
    ["Pol. accuracy", "Constant", "Variable", "Variable"],
    ["Spectropolarimetry", "Yes (SP)", "CRISP", "ViSP/DL-NIRSP"],
    ["Year", "2006", "2002", "2020"],
]

table = ax.table(cellText=data, colLabels=columns, loc="center",
                 cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

# Color header
for j in range(len(columns)):
    table[0, j].set_facecolor("#E0E0E0")

ax.set_title("Telescope Comparison", pad=20)

plt.tight_layout()
plt.show()

print("Key SOT advantage:")
print("  Resolution: lower than SST/DKIST")
print("  BUT: stable PSF + continuous observation + constant polarimetry")
print("  → Enables science impossible from ground:")
print("    - Multi-day tracking of magnetic flux emergence")
print("    - Lagrangian tracking of individual flux elements")
print("    - Precise polarimetry without atmospheric crosstalk")

## Part 9: SOT 데이터 제품과 데이터량 / SOT Data Products and Volume

SOT는 BFI, NFI, SP의 3개 검출기에서 다양한 데이터 제품을 생성합니다.
SOT generates various data products from its three detectors: BFI, NFI, and SP.

텔레메트리 제약 때문에 온보드 처리(Stokes 복조, 데이터 압축)가 필수적입니다.
Telemetry constraints make onboard processing (Stokes demodulation, data compression) essential.

In [ ]:
# =============================================================
# SOT Data Products and Telemetry Budget
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Data product summary
ax = axes[0]
ax.axis("off")

products = [
    ["BFI Filtergram", "4K×2K", '0.054"', "<10 s", "Photometry"],
    ["NFI Filtergram", "4K×2K", '0.08"', "Variable", "Spectral imaging"],
    ["Dopplergram", "4K×2K", '0.08"', "~10 s", "Velocity"],
    ["LOS Magnetogram", "4K×2K", '0.08"', "~20 s", "B_LOS"],
    ["Stokes IQUV", "4K×2K", '0.08"', "1.6-4.8 s", "Vector B"],
    ["SP Normal Map", "1K×1K", '0.15"×0.16"', "83 min", "Full Stokes spectra"],
    ["SP Fast Map", "1K×1K", '0.30"×0.32"', "30 min", "Full Stokes spectra"],
]

columns = ["Product", "Format", "Pixel", "Cadence", "Content"]
table = ax.table(cellText=products, colLabels=columns, loc="center",
                 cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.5)
for j in range(len(columns)):
    table[0, j].set_facecolor("#E0E0E0")
ax.set_title("SOT Data Products", fontsize=12, pad=20)

# Right: Data compression pipeline
ax = axes[1]

# Raw data rate calculation
bfi_pixels = 4096 * 2048
nfi_pixels = 4096 * 2048
bits_per_pixel = 16  # raw CCD output

# Different compression stages
stages = [
    "Raw CCD\n(16-bit)",
    "Bit compression\n(16→12 bit)",
    "JPEG-like\n(~3 bit/pix)",
    "Stokes demod\n(~1.5 bit/pix)",
]
bits_per_pix = [16, 12, 3, 1.5]
data_rates = [b * bfi_pixels / 1e9 for b in bits_per_pix]  # Gbit per frame

colors_comp = ["#E91E63", "#FF9800", "#4CAF50", "#2196F3"]
bars = ax.barh(range(len(stages)), bits_per_pix, color=colors_comp,
               edgecolor="black", height=0.5)
ax.set_yticks(range(len(stages)))
ax.set_yticklabels(stages, fontsize=9)
ax.set_xlabel("Bits per Pixel")
ax.set_title("SOT Data Compression Pipeline")

for bar, val in zip(bars, bits_per_pix):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            "{:.1f} bit/pix".format(val), va="center")

plt.tight_layout()
plt.show()

# Telemetry budget calculation
print("SOT Telemetry Budget:")
print("=" * 50)
print("  BFI/NFI CCD: 4096 × 2048 pixels")
print("  Raw frame: {:.1f} MB".format(bfi_pixels * 16 / 8 / 1e6))
print("")
print("  Compression ratios:")
print("    16→12 bit depth: {:.1f}x".format(16 / 12))
print("    JPEG filtergram: {:.1f}x".format(16 / 3))
print("    Stokes demod:    {:.1f}x".format(16 / 1.5))
print("")
print("  Hinode total downlink: ~2 Mbps (allocated to SOT+XRT+EIS)")
print("  ESA Svalbard station: nearly every-orbit downlink")
print("  → Enables high-cadence, wide-FOV observations")

## Part 10: SOT 기기 발전사 / SOT Instrument Heritage Timeline

Hinode/SOT는 우주 태양 광학 관측의 새로운 이정표입니다.
Hinode/SOT represents a new milestone in space-based solar optical observation.

이전 미션과의 비교를 통해 SOT의 위치를 확인합니다.
We compare SOT with previous missions to understand its place in history.

In [ ]:
# =============================================================
# Solar Space Telescope Evolution
# =============================================================

missions = [
    ("OSO-1", 1962, 0, "First solar space obs."),
    ("Skylab/ATM", 1973, 32, "UV/X-ray imaging"),
    ("SMM", 1980, 15, "Flare observation"),
    ("Yohkoh", 1991, 25, "Soft X-ray (NAOJ)"),
    ("SOHO", 1995, 20, "L1 multi-instrument"),
    ("TRACE", 1998, 30, "EUV high-res"),
    ("Hinode/SOT", 2006, 50, "★ Optical spectropolarimetry"),
    ("SDO", 2010, 14, "Full-disk magnetograph"),
    ("IRIS", 2013, 20, "UV spectrograph"),
]

years = [m[1] for m in missions]
apertures = [m[2] for m in missions]
names = [m[0] for m in missions]
descriptions = [m[3] for m in missions]

fig, axes = plt.subplots(2, 1, figsize=(14, 10),
                         gridspec_kw={"height_ratios": [1, 1.3]})

# Top: Timeline
ax = axes[0]
ax.set_xlim(1958, 2018)
ax.set_ylim(-1.5, 1.5)
ax.axhline(y=0, color="black", linewidth=2)

for i, (name, year, aper, desc) in enumerate(missions):
    is_sot = "SOT" in name
    color = "#E91E63" if is_sot else "#2196F3"
    ms = 15 if is_sot else 8
    ax.plot(year, 0, "o", color=color, markersize=ms, zorder=5)

    y_off = 0.7 if i % 2 == 0 else -0.9
    fontweight = "bold" if is_sot else "normal"
    ax.annotate(
        "{}\n({})".format(name, year),
        xy=(year, 0),
        xytext=(year, y_off),
        fontsize=8,
        fontweight=fontweight,
        ha="center",
        arrowprops=dict(arrowstyle="-", color="gray", alpha=0.5),
    )

ax.set_yticks([])
ax.set_xlabel("Year")
ax.set_title("Solar Space Telescope Timeline")

# Bottom: Aperture comparison
ax = axes[1]
colors_bar = ["#E91E63" if "SOT" in n else "#90CAF9" for n in names]
bars = ax.bar(range(len(missions)), apertures, color=colors_bar,
              edgecolor="black", width=0.6)

for i, (bar, val) in enumerate(zip(bars, apertures)):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                "{} cm".format(val), ha="center", fontsize=8,
                fontweight="bold" if "SOT" in names[i] else "normal")

ax.set_xticks(range(len(missions)))
xlabels = ["{}\n({})".format(n, y) for n, y in zip(names, years)]
ax.set_xticklabels(xlabels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("Optical Aperture [cm]")
ax.set_title("Optical Aperture Comparison (Optical/UV Telescopes)")

plt.tight_layout()
plt.show()

print("Hinode/SOT Milestones:")
print("  ★ Largest solar optical telescope in space (50 cm)")
print("  ★ First diffraction-limited optical telescope in space")
print("  ★ First space-based Stokes spectropolarimetry")
print("  ★ Image stabilization: 0.007\" rms (4× better than spec)")
print("  ★ Continuous high-res observations for months")
print("")
print("Scientific legacy (by 2008 publication):")
print("  - Ubiquitous horizontal magnetic fields in quiet Sun")
print("  - Chromospheric jets and Type II spicules")
print("  - Sunspot fine structure and penumbral dynamics")
print("  - Convective collapse of magnetic flux tubes")